# Détection de fraude bancaire — Phase 2 : Preprocessing
---
**Notebook :** `02_preprocessing.ipynb`  
**Auteur :** kgueye  
**Dataset :** [Credit Card Fraud Detection — ULB](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Prérequis :** `01_eda.ipynb`

---

## Objectifs de ce notebook

À partir des conclusions de l'EDA, ce notebook construit le pipeline de
transformation complet avant la modélisation :

1. **Scaling** — `log1p` + `StandardScaler` sur `Amount`, `StandardScaler` sur `Time`
2. **Feature engineering** — ajout de `Hour`, suppression de `Time` brut
3. **Split train/test** — stratifié pour respecter le déséquilibre
4. **Gestion du déséquilibre** — comparaison `class_weight` vs `SMOTE`
5. **Export** — sauvegarde des datasets prêts pour la modélisation

## Inputs / Outputs

| | Fichier |
|---|---|
| Input | `/kaggle/input/creditcardfraud/creditcard.csv` |
| Output 1 | `X_train.csv`, `X_test.csv` |
| Output 2 | `y_train.csv`, `y_test.csv` |
| Output 3 | `X_train_smote.csv`, `y_train_smote.csv` |

## 0. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# Reproductibilité
RANDOM_STATE = 42

print("Librairies chargées.")
print(f"Random state : {RANDOM_STATE}")

Librairies chargées.
Random state : 42


## 1. Chargement des données

In [2]:
df = pd.read_csv("/kaggle/input/datasets/kgueye/creditcardfraud/creditcard.csv")

print(f"Shape : {df.shape}")
print(f"Fraudes : {df['Class'].sum()} ({df['Class'].mean()*100:.3f}%)")

Shape : (284807, 31)
Fraudes : 492 (0.173%)


## 2. Feature engineering

In [3]:
# Extraction de l'heure depuis Time
df['Hour'] = (df['Time'] % 86400) // 3600

# log1p sur Amount — réduit l'asymétrie de la distribution
df['Amount_log'] = np.log1p(df['Amount'])

print("Features créées :")
print(f"  Hour    — min: {df['Hour'].min():.0f}h  max: {df['Hour'].max():.0f}h")
print(f"  Amount_log — min: {df['Amount_log'].min():.3f}  max: {df['Amount_log'].max():.3f}")
print(f"\nShape après feature engineering : {df.shape}")

Features créées :
  Hour    — min: 0h  max: 23h
  Amount_log — min: 0.000  max: 10.154

Shape après feature engineering : (284807, 33)


### Observation
Deux features créées à partir des colonnes brutes :

- `Hour` : extraite de `Time` via `Time % 86400 // 3600` — l'EDA a montré un pic
  de fraude à 2h–3h du matin, cette feature capture ce signal temporel
- `Amount_log` : `log1p(Amount)` compresse la longue queue droite de la distribution
  des montants — les algorithmes sensibles aux échelles (régression logistique,
  SVM) en bénéficient directement

`Time` et `Amount` bruts seront supprimés après le scaling — on garde uniquement
les versions transformées.

## 3. Scaling

In [4]:
scaler_amount = StandardScaler()
scaler_hour   = StandardScaler()

df['Amount_scaled'] = scaler_amount.fit_transform(df[['Amount_log']])
df['Hour_scaled']   = scaler_hour.fit_transform(df[['Hour']])

# Suppression des colonnes brutes remplacées
df_clean = df.drop(columns=['Time', 'Amount', 'Amount_log', 'Hour'])

print("Colonnes finales :")
print(df_clean.columns.tolist())
print(f"\nShape finale : {df_clean.shape}")
print(f"\nVérification scaling Amount_scaled :")
print(f"  mean : {df_clean['Amount_scaled'].mean():.4f}  (attendu ≈ 0)")
print(f"  std  : {df_clean['Amount_scaled'].std():.4f}   (attendu ≈ 1)")

Colonnes finales :
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_scaled', 'Hour_scaled']

Shape finale : (284807, 31)

Vérification scaling Amount_scaled :
  mean : 0.0000  (attendu ≈ 0)
  std  : 1.0000   (attendu ≈ 1)


### Observation
Le `StandardScaler` transforme chaque feature pour avoir **moyenne = 0** et
**écart-type = 1`. C'est essentiel pour deux raisons :

- Les features V1–V28 sont déjà normalisées via PCA — `Amount` et `Hour` doivent
  être sur la même échelle pour ne pas dominer les calculs de distance
- La régression logistique et les SVM sont particulièrement sensibles aux échelles —
  sans scaling, `Amount` brut (0–25 691) écraserait les V1–V28 (−30 à +30)

**Pourquoi `fit_transform` sur tout le dataset ici ?**
On scale d'abord, on split ensuite — dans un projet de production ce serait
l'inverse (`fit` sur train uniquement, `transform` sur test).
Ici le dataset est statique et non temporel donc les deux approches donnent
des résultats quasi identiques, mais on le notera explicitement.

## 4. Split train / test

In [5]:
X = df_clean.drop(columns=['Class'])
y = df_clean['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y        # conserve le ratio fraude/légitime dans chaque split
)

print("=== Dimensions ===")
print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")

print("\n=== Vérification du stratify ===")
print(f"Taux fraude train : {y_train.mean()*100:.3f}%")
print(f"Taux fraude test  : {y_test.mean()*100:.3f}%")
print(f"Taux fraude total : {y.mean()*100:.3f}%")

=== Dimensions ===
X_train : (227845, 30)  |  y_train : (227845,)
X_test  : (56962, 30)   |  y_test  : (56962,)

=== Vérification du stratify ===
Taux fraude train : 0.173%
Taux fraude test  : 0.172%
Taux fraude total : 0.173%


### Observation
**Pourquoi `stratify=y` est obligatoire ici ?**

Avec 492 fraudes sur 284 807 transactions, un split aléatoire classique risque
de mal répartir les fraudes — on pourrait par exemple se retrouver avec
80% des fraudes dans le train et 20% seulement dans le test, faussant
complètement l'évaluation.

`stratify=y` garantit que le ratio 0.173% est conservé **exactement**
dans le train et dans le test. C'est une règle systématique pour
tout dataset déséquilibré.

**Règle du 80/20 :**
- 80% en train → le modèle apprend sur ces données
- 20% en test → on évalue sur des données jamais vues
- Le test ne sera **jamais touché** avant l'évaluation finale —
  pas de scaling, pas de SMOTE dessus

## 5. Gestion du déséquilibre — SMOTE

In [6]:
print("=== Avant SMOTE ===")
print(f"Légitimes train : {(y_train == 0).sum():,}")
print(f"Fraudes train   : {(y_train == 1).sum():,}")
print(f"Ratio           : 1 fraude / {(y_train==0).sum() // (y_train==1).sum()} légitimes")

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\n=== Après SMOTE ===")
print(f"Légitimes train : {(y_train_smote == 0).sum():,}")
print(f"Fraudes train   : {(y_train_smote == 1).sum():,}")
print(f"Ratio           : 1 fraude / {(y_train_smote==0).sum() // (y_train_smote==1).sum()} légitime")
print(f"\nShape X_train_smote : {X_train_smote.shape}")

=== Avant SMOTE ===
Légitimes train : 227,451
Fraudes train   : 394
Ratio           : 1 fraude / 577 légitimes

=== Après SMOTE ===
Légitimes train : 227,451
Fraudes train   : 227,451
Ratio           : 1 fraude / 1 légitime

Shape X_train_smote : (454902, 30)


### Observation
**SMOTE — Synthetic Minority Oversampling Technique**

SMOTE ne copie pas simplement les fraudes existantes (ce serait du oversampling
naïf). Il **crée de nouvelles fraudes synthétiques** en interpolant entre
des fraudes existantes proches dans l'espace des features.

Concrètement : pour chaque fraude réelle, SMOTE cherche ses k voisins les plus
proches parmi les autres fraudes, puis génère un nouveau point aléatoire
sur le segment entre les deux. Le résultat est une fraude plausible
mais jamais vue dans les données originales.

**Résultat ici :**
- Avant SMOTE : 394 fraudes pour 227 451 légitimes → ratio 1/577
- Après SMOTE : 227 451 fraudes pour 227 451 légitimes → ratio 1/1
- SMOTE a généré **227 057 fraudes synthétiques** pour rééquilibrer parfaitement

**Attention — ratio 1/1 est-il optimal ?**
Un ratio parfait 50/50 n'est pas toujours le meilleur choix. On peut passer
`sampling_strategy=0.1` à SMOTE pour avoir 1 fraude / 10 légitimes — moins
agressif, parfois meilleur en pratique. On testera les deux en modélisation.

**Pourquoi SMOTE uniquement sur le train ?**
Appliquer SMOTE sur le test serait une **fuite de données** — on évalue toujours
sur des données 100% réelles, jamais synthétiques.
Le test reste intact : 56 962 transactions réelles, 98 fraudes réelles.

## 6. Export des datasets

In [7]:
import os
os.makedirs("/kaggle/working/data", exist_ok=True)

# Datasets standard
X_train.to_csv("/kaggle/working/data/X_train.csv", index=False)
X_test.to_csv("/kaggle/working/data/X_test.csv",   index=False)
y_train.to_csv("/kaggle/working/data/y_train.csv", index=False)
y_test.to_csv("/kaggle/working/data/y_test.csv",   index=False)

# Datasets SMOTE
pd.DataFrame(X_train_smote, columns=X_train.columns).to_csv(
    "/kaggle/working/data/X_train_smote.csv", index=False)
pd.Series(y_train_smote, name='Class').to_csv(
    "/kaggle/working/data/y_train_smote.csv", index=False)

print("=== Fichiers exportés ===")
for f in os.listdir("/kaggle/working/data"):
    size = os.path.getsize(f"/kaggle/working/data/{f}") / 1024 / 1024
    print(f"  {f:<30} {size:.1f} MB")

=== Fichiers exportés ===
  X_test.csv                     30.0 MB
  y_train.csv                    0.4 MB
  y_train_smote.csv              0.9 MB
  X_train_smote.csv              246.3 MB
  X_train.csv                    120.1 MB
  y_test.csv                     0.1 MB


### Observation
Les 6 fichiers exportés couvrent deux scénarios qu'on comparera en modélisation :

| Scénario | Train utilisé | Approche |
|---|---|---|
| A | `X_train` + `y_train` | `class_weight='balanced'` dans le modèle |
| B | `X_train_smote` + `y_train_smote` | Données rééquilibrées par SMOTE |

Dans les deux cas le **test ne change pas** — `X_test` et `y_test` sont
toujours les données réelles originales pour une évaluation honnête.